# Scenario: Automated Data Analysis Assistant (Corporate Use Case)
🎯 Problem Statement

 In companies:

 Analysts manually write Python code for data analysis

 It takes time to compute statistics and validate results

 👉 Solution:
 A multi-agent AutoGen system where:

 🤖 Assistant writes Python code

 ⚙️ UserProxy executes it automatically

 🔁 System iterates until correct output is achieved

 🔄 Workflow
 Business Query → AI writes code → Code executes → Results validated → Final output
 Clean conflicting packages

In [1]:
!pip install numpy==1.26.4 pyautogen==0.2.25

In [2]:
import autogen
print("✅ AutoGen installed successfully")

✅ AutoGen installed successfully


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [9]:
# 🔹 Install (run once)
import sys
!{sys.executable} -m pip install pyautogen openai

# 🔹 Imports
import autogen
from getpass import getpass

# 🔐 Step 1: Ask API Key (at runtime)
groq_api_key = getpass("Enter your GROQ API Key: ").strip()

# 🔍 Debug (optional)
print("Key length:", len(groq_api_key))

# 🔹 Step 2: LLM Config (AFTER taking key)
config_list = [
    {
        "model": "llama-3.3-70b-versatile",   # stable Groq model
        "api_key": groq_api_key,
        "base_url": "https://api.groq.com/openai/v1"
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0
}

# 🔹 Step 3: Assistant Agent
assistant = autogen.AssistantAgent(
    name="Data_Analyst",
    llm_config=llm_config,
    system_message="Write Python code for business analysis and explain results."
)

# 🔹 Step 4: User Proxy (Executor)
user_proxy = autogen.UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config={
        "work_dir": "coding",
        "use_docker": False
    }
)

# 🔹 Step 5: Run Task
user_proxy.initiate_chat(
    assistant,
    message="""
Analyze sales data: [12000, 15000, 10000, 18000, 13000]

Find:
- Mean
- Median
- Mode
- Highest & Lowest
- Business insights
"""
)

Enter your GROQ API Key: ··········
Key length: 56
Executor (to Data_Analyst):


Analyze sales data: [12000, 15000, 10000, 18000, 13000]

Find:
- Mean
- Median
- Mode
- Highest & Lowest
- Business insights


--------------------------------------------------------------------------------
Data_Analyst (to Executor):

### Sales Data Analysis
#### Introduction
The following Python code analyzes the given sales data to calculate the mean, median, mode, highest, and lowest sales values. Additionally, it provides business insights based on the calculated metrics.

#### Code
```python
import statistics

# Given sales data
sales_data = [12000, 15000, 10000, 18000, 13000]

# Calculate mean
mean_sales = statistics.mean(sales_data)

# Calculate median
median_sales = statistics.median(sales_data)

# Calculate mode
mode_sales = statistics.mode(sales_data)

# Find highest and lowest sales
highest_sales = max(sales_data)
lowest_sales = min(sales_data)

# Print results
print("Sales Data Analysis:")
pr

ChatResult(chat_id=None, chat_history=[{'content': '\nAnalyze sales data: [12000, 15000, 10000, 18000, 13000]\n\nFind:\n- Mean\n- Median\n- Mode\n- Highest & Lowest\n- Business insights\n', 'role': 'assistant'}, {'content': '### Sales Data Analysis\n#### Introduction\nThe following Python code analyzes the given sales data to calculate the mean, median, mode, highest, and lowest sales values. Additionally, it provides business insights based on the calculated metrics.\n\n#### Code\n```python\nimport statistics\n\n# Given sales data\nsales_data = [12000, 15000, 10000, 18000, 13000]\n\n# Calculate mean\nmean_sales = statistics.mean(sales_data)\n\n# Calculate median\nmedian_sales = statistics.median(sales_data)\n\n# Calculate mode\nmode_sales = statistics.mode(sales_data)\n\n# Find highest and lowest sales\nhighest_sales = max(sales_data)\nlowest_sales = min(sales_data)\n\n# Print results\nprint("Sales Data Analysis:")\nprint("---------------------")\nprint(f"Mean Sales: ${mean_sales:.2f}

# Scenario: AI Code Review & Execution System (Corporate Dev Team)
 🎯 Problem Statement

 In companies:

 Developers write code

 Reviewers check for bugs

 Testing is manual

 👉 Solution:
 A multi-agent system where:

 👨‍💻 Coder Agent writes code

 🔍 Reviewer Agent validates + executes code

 🔁 Loop continues until correct



In [10]:
# 🔹 Install (run once)
import sys
!{sys.executable} -m pip install pyautogen openai

# 🔹 Imports
import autogen
from getpass import getpass

# -----------------------------
# 🔐 Ask API Key at runtime (END)
# -----------------------------
groq_api_key = getpass("Enter your GROQ API Key: ").strip()

# -----------------------------
# LLM Config (Groq)
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",  # ✅ active model
            "api_key": groq_api_key,
            "base_url": "https://api.groq.com/openai/v1"
        }
    ],
    "temperature": 0
}

# -----------------------------
# Agent 1: Coder
# -----------------------------
coder = autogen.AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message="""You are an expert Python developer.

Write clean, well-commented code.

Rules:
- Handle edge cases
- Add test cases
- Ensure code runs without errors
- End your response with DONE"""
)

# -----------------------------
# Agent 2: Reviewer (Executor)
# -----------------------------
reviewer = autogen.UserProxyAgent(
    name="Reviewer",
    human_input_mode="NEVER",
    code_execution_config={
        "work_dir": "review",
        "use_docker": False
    },
    is_termination_msg=lambda x: "DONE" in x.get("content", ""),
    system_message="""You are a code reviewer.

Responsibilities:
1. Review the code for bugs
2. Execute the code
3. Validate outputs
4. Ask for fixes if incorrect
5. Approve when correct"""
)

# -----------------------------
# Start Collaboration
# -----------------------------
reviewer.initiate_chat(
    coder,
    message="""Create a Python function that:
- Generates Fibonacci sequence up to N
- Returns it as a list
- Handles edge cases (N=0, negative N)

Test it with N=10 and print output"""
)

Enter your GROQ API Key: ··········
Reviewer (to Coder):

Create a Python function that:
- Generates Fibonacci sequence up to N
- Returns it as a list
- Handles edge cases (N=0, negative N)

Test it with N=10 and print output

--------------------------------------------------------------------------------
Coder (to Reviewer):

```python
def generate_fibonacci(n):
    """
    Generates Fibonacci sequence up to N.

    Args:
        n (int): The number of terms in the Fibonacci sequence.

    Returns:
        list: A list of Fibonacci numbers up to N.

    Raises:
        ValueError: If N is a negative integer.
    """
    # Handle edge cases
    if n < 0:
        raise ValueError("N must be a non-negative integer.")
    elif n == 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    # Initialize the Fibonacci sequence
    fib_sequence = [0, 1]

    # Generate the Fibonacci sequence up to N
    while len(fib_sequence) < n:
        fib_seque

ChatResult(chat_id=None, chat_history=[{'content': 'Create a Python function that:\n- Generates Fibonacci sequence up to N\n- Returns it as a list\n- Handles edge cases (N=0, negative N)\n\nTest it with N=10 and print output', 'role': 'assistant'}, {'content': '```python\ndef generate_fibonacci(n):\n    """\n    Generates Fibonacci sequence up to N.\n\n    Args:\n        n (int): The number of terms in the Fibonacci sequence.\n\n    Returns:\n        list: A list of Fibonacci numbers up to N.\n\n    Raises:\n        ValueError: If N is a negative integer.\n    """\n    # Handle edge cases\n    if n < 0:\n        raise ValueError("N must be a non-negative integer.")\n    elif n == 0:\n        return []\n    elif n == 1:\n        return [0]\n    elif n == 2:\n        return [0, 1]\n\n    # Initialize the Fibonacci sequence\n    fib_sequence = [0, 1]\n\n    # Generate the Fibonacci sequence up to N\n    while len(fib_sequence) < n:\n        fib_sequence.append(fib_sequence[-1] + fib_seque

# Scenario: Sales Data Analysis with AutoGen Agents
 Characters:
 - Data_Analyst (Assistant Agent) → An AI that writes Python code and explains business results.
 - Executor (UserProxy Agent) → Runs the code and reports back the outputs.


In [11]:
# 🔹 Install (run once)
import sys
!{sys.executable} -m pip install pyautogen openai

# 🔹 Imports
import autogen
from getpass import getpass

# -----------------------------
# 🔧 Step 1: LLM Config (empty initially)
# -----------------------------
config_list = [
    {
        "model": "llama-3.3-70b-versatile",
        "api_key": "",   # placeholder
        "base_url": "https://api.groq.com/openai/v1"
    }
]

llm_config = {"config_list": config_list, "temperature": 0}

# -----------------------------
# 🤖 Step 2: Create Assistant Agent
# -----------------------------
assistant = autogen.AssistantAgent(
    name="Data_Analyst",
    llm_config=llm_config,
    system_message="Write Python code for business analysis and explain results."
)

# -----------------------------
# ⚙️ Step 3: Create Executor Agent
# -----------------------------
user_proxy = autogen.UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config={"work_dir": "coding", "use_docker": False}
)

# -----------------------------
# 🔐 Step 4: Ask API key at runtime (END)
# -----------------------------
groq_api_key = getpass("Enter your GROQ API Key: ").strip()

# Inject API key into config
assistant.llm_config["config_list"][0]["api_key"] = groq_api_key

# -----------------------------
# 📊 Step 5: Run Task
# -----------------------------
user_proxy.initiate_chat(
    assistant,
    message="""
Analyze sales data: [12000, 15000, 10000, 18000, 13000]

Find:
- Mean
- Median
- Mode
- Highest & Lowest
- Business insights
"""
)

Enter your GROQ API Key: ··········
Executor (to Data_Analyst):


Analyze sales data: [12000, 15000, 10000, 18000, 13000]

Find:
- Mean
- Median
- Mode
- Highest & Lowest
- Business insights


--------------------------------------------------------------------------------
Data_Analyst (to Executor):

### Sales Data Analysis
#### Introduction
The following Python code analyzes the given sales data to calculate the mean, median, mode, highest, and lowest sales values. Additionally, it provides business insights based on the calculated metrics.

#### Code
```python
import statistics

# Given sales data
sales_data = [12000, 15000, 10000, 18000, 13000]

# Calculate mean
mean_sales = statistics.mean(sales_data)

# Calculate median
median_sales = statistics.median(sales_data)

# Calculate mode
mode_sales = statistics.mode(sales_data)

# Find highest and lowest sales
highest_sales = max(sales_data)
lowest_sales = min(sales_data)

# Print results
print("Sales Data Analysis:")
print("----------

ChatResult(chat_id=None, chat_history=[{'content': '\nAnalyze sales data: [12000, 15000, 10000, 18000, 13000]\n\nFind:\n- Mean\n- Median\n- Mode\n- Highest & Lowest\n- Business insights\n', 'role': 'assistant'}, {'content': '### Sales Data Analysis\n#### Introduction\nThe following Python code analyzes the given sales data to calculate the mean, median, mode, highest, and lowest sales values. Additionally, it provides business insights based on the calculated metrics.\n\n#### Code\n```python\nimport statistics\n\n# Given sales data\nsales_data = [12000, 15000, 10000, 18000, 13000]\n\n# Calculate mean\nmean_sales = statistics.mean(sales_data)\n\n# Calculate median\nmedian_sales = statistics.median(sales_data)\n\n# Calculate mode\nmode_sales = statistics.mode(sales_data)\n\n# Find highest and lowest sales\nhighest_sales = max(sales_data)\nlowest_sales = min(sales_data)\n\n# Print results\nprint("Sales Data Analysis:")\nprint("---------------------")\nprint(f"Mean Sales: ${mean_sales:.2f}

# Scenario: Automated Business Report Assistant
🎯 Problem Statement
In companies:
- Managers spend hours compiling data from multiple sources (CRM, ERP, spreadsheets).
- Reports often require manual formatting, chart creation, and consistency checks.
- This slows down decision-making and introduces human error.
👉 Solution
A multi-agent AutoGen system where:
- 🤖 Assistant Agent drafts report templates and queries relevant data.
- ⚙️ UserProxy Agent executes data pulls and formatting automatically.
- 🔁 The system iterates until the report meets formatting, accuracy, and completeness standards.
🔄 Workflow
Business Request → AI drafts report structure → Data fetched & formatted → Charts auto-generated → Validation loop → Final polished report

In [12]:
# 🔹 Install (run once)
import sys
!{sys.executable} -m pip install pyautogen openai pandas matplotlib

# 🔹 Imports
import autogen
from getpass import getpass

# 🔐 Ask API Key at runtime
groq_api_key = getpass("Enter your GROQ API Key: ").strip()

# -----------------------------
# 🔧 LLM Config
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": groq_api_key,
            "base_url": "https://api.groq.com/openai/v1"
        }
    ],
    "temperature": 0
}

# -----------------------------
# 🤖 Agent 1: Report Generator
# -----------------------------
report_agent = autogen.AssistantAgent(
    name="ReportAgent",
    llm_config=llm_config,
    system_message="""
You are a Business Analyst AI.

Your job:
1. Create structured reports
2. Generate Python code for:
   - Data analysis
   - Summary statistics
   - Charts (matplotlib)
3. Ensure:
   - Clean formatting
   - Business insights
   - Actionable recommendations

Always:
- Write executable Python code
- Print results clearly
- Generate charts
- End with DONE
"""
)

# -----------------------------
# ⚙️ Agent 2: Executor
# -----------------------------
executor = autogen.UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=6,
    code_execution_config={
        "work_dir": "reports",
        "use_docker": False
    },
    is_termination_msg=lambda x: "DONE" in x.get("content", "")
)

# -----------------------------
# 📊 Business Task
# -----------------------------
task = """
You are given sales data:

Region A: [12000, 15000, 18000]
Region B: [10000, 13000, 9000]
Region C: [20000, 22000, 21000]

Create a BUSINESS REPORT with:

1. Summary statistics (mean, growth)
2. Comparison between regions
3. Identify best & worst region
4. Generate bar chart
5. Provide business insights
6. Recommend actions

Ensure clean formatting and explanation.
"""

# -----------------------------
# 🚀 Run System
# -----------------------------
executor.initiate_chat(report_agent, message=task)

Enter your GROQ API Key: ··········
Executor (to ReportAgent):


You are given sales data:

Region A: [12000, 15000, 18000]
Region B: [10000, 13000, 9000]
Region C: [20000, 22000, 21000]

Create a BUSINESS REPORT with:

1. Summary statistics (mean, growth)
2. Comparison between regions
3. Identify best & worst region
4. Generate bar chart
5. Provide business insights
6. Recommend actions

Ensure clean formatting and explanation.


--------------------------------------------------------------------------------
ReportAgent (to Executor):

**Business Report: Sales Data Analysis**

### Introduction

This report provides an analysis of sales data from three regions: A, B, and C. The data is presented in a structured format, and key statistics are calculated to provide insights into the performance of each region.

### Summary Statistics

The following table summarizes the sales data for each region:

| Region | Sales Data | Mean | Growth |
| --- | --- | --- | --- |
| A | [12000, 15000, 180

ChatResult(chat_id=None, chat_history=[{'content': '\nYou are given sales data:\n\nRegion A: [12000, 15000, 18000]\nRegion B: [10000, 13000, 9000]\nRegion C: [20000, 22000, 21000]\n\nCreate a BUSINESS REPORT with:\n\n1. Summary statistics (mean, growth)\n2. Comparison between regions\n3. Identify best & worst region\n4. Generate bar chart\n5. Provide business insights\n6. Recommend actions\n\nEnsure clean formatting and explanation.\n', 'role': 'assistant'}, {'content': '**Business Report: Sales Data Analysis**\n=====================================\n\n### Introduction\n\nThis report provides an analysis of sales data from three regions: A, B, and C. The data is presented in a structured format, and key statistics are calculated to provide insights into the performance of each region.\n\n### Summary Statistics\n\nThe following table summarizes the sales data for each region:\n\n| Region | Sales Data | Mean | Growth |\n| --- | --- | --- | --- |\n| A | [12000, 15000, 18000] | 15000.0 | 5

# Scenario: AI Marketing Content & Validation System (Corporate Marketing Team)
🎯 Problem Statement
In companies:
- Marketing teams draft campaign content manually.
- Editors check for tone, compliance, and brand alignment.
- Testing across platforms (emails, social posts, ads) is manual and time-consuming.
👉 Solution
A multi-agent system where:
- ✍️ Content Creator Agent generates marketing copy, visuals, or ad text.
- 🔎 Validator Agent reviews for compliance, tone, and brand guidelines, then simulates deployment.
- 🔁 The loop continues until the content meets quality and compliance standards.
🔄 Workflow
Campaign Request → AI drafts content → Validator checks tone/compliance → Test deployment simulation → Iteration loop → Final approved campaign assets
🧹 Conflict Resolution
- Resolve conflicting brand guidelines (e.g., tone mismatch between product lines).
- Standardize formatting across channels (social, email, ads).
- Ensure package/library consistency for creative assets (e.g., Canva API vs. Adobe API).

In [14]:
# 🔹 Install (run once)
import sys
!{sys.executable} -m pip install pyautogen openai

# 🔹 Imports
import autogen
from getpass import getpass

# 🔐 API Key (ask at runtime)
groq_api_key = getpass("Enter your GROQ API Key: ").strip()

# -----------------------------
# 🔧 LLM Config
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": groq_api_key,
            "base_url": "https://api.groq.com/openai/v1"
        }
    ],
    "temperature": 0.7   # creativity for marketing
}

# -----------------------------
# ✍️ Agent 1: Content Creator
# -----------------------------
creator = autogen.AssistantAgent(
    name="ContentCreator",
    llm_config=llm_config,
    system_message="""
You are a Marketing Content Creator.

Tasks:
1. Generate campaign content:
   - Email copy
   - Social media post
   - Ad headline + description
2. Maintain:
   - Engaging tone
   - Brand consistency (professional and friendly)
3. Provide variations if needed

Output format:
- Email
- Instagram Post
- Ad Copy

End with: DRAFT_READY
"""
)

# -----------------------------
# 🔎 Agent 2: Validator
# -----------------------------
validator = autogen.AssistantAgent(
    name="Validator",
    llm_config=llm_config,
    system_message="""
You are a Marketing Compliance & Quality Validator.

Responsibilities:
1. Check tone (brand: professional + friendly)
2. Check compliance (no misleading claims)
3. Ensure consistency across:
   - Email
   - Social
   - Ads
4. Resolve conflicts in tone or messaging
5. Simulate deployment readiness

If issues found:
- Explain clearly
- Ask for improvements

If perfect:
Return: APPROVED
"""
)

# -----------------------------
# ⚙️ Agent 3: Executor
# -----------------------------
executor = autogen.UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=8,
    code_execution_config={"work_dir": "marketing", "use_docker": False},
    is_termination_msg=lambda x: "APPROVED" in x.get("content", "")
)

# -----------------------------
# 👥 Group Chat Setup
# -----------------------------
groupchat = autogen.GroupChat(
    agents=[executor, creator, validator],
    messages=[],
    max_round=10,
    speaker_selection_method="auto"
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

# -----------------------------
# 🚀 Campaign Request
# -----------------------------
task = """
Create a marketing campaign for a new AI-powered learning app.

Requirements:
- Target audience: college students
- Tone: friendly, motivating, slightly premium
- Channels:
  1. Email campaign
  2. Instagram post
  3. Google Ad copy

Constraints:
- No exaggerated claims
- Maintain consistent messaging
- Keep branding aligned

Run validation and iterate until APPROVED.
"""

# -----------------------------
# ▶️ Run System
# -----------------------------
executor.initiate_chat(manager, message=task)

Enter your GROQ API Key: ··········
Executor (to chat_manager):


Create a marketing campaign for a new AI-powered learning app.

Requirements:
- Target audience: college students
- Tone: friendly, motivating, slightly premium
- Channels:
  1. Email campaign
  2. Instagram post
  3. Google Ad copy

Constraints:
- No exaggerated claims
- Maintain consistent messaging
- Keep branding aligned

Run validation and iterate until APPROVED.


--------------------------------------------------------------------------------
ContentCreator (to chat_manager):

**Email Campaign**

Subject: Unlock Your Potential with Our AI-Powered Learning App

Dear [Student],

As a college student, you're constantly looking for ways to stay ahead of the curve and achieve your academic goals. That's why we're excited to introduce you to our new AI-powered learning app, designed to help you learn more efficiently and effectively.

Our app uses cutting-edge technology to provide personalized learning experiences, tai

ChatResult(chat_id=None, chat_history=[{'content': '\nCreate a marketing campaign for a new AI-powered learning app.\n\nRequirements:\n- Target audience: college students\n- Tone: friendly, motivating, slightly premium\n- Channels:\n  1. Email campaign\n  2. Instagram post\n  3. Google Ad copy\n\nConstraints:\n- No exaggerated claims\n- Maintain consistent messaging\n- Keep branding aligned\n\nRun validation and iterate until APPROVED.\n', 'role': 'assistant'}, {'content': "**Email Campaign**\n\nSubject: Unlock Your Potential with Our AI-Powered Learning App\n\nDear [Student],\n\nAs a college student, you're constantly looking for ways to stay ahead of the curve and achieve your academic goals. That's why we're excited to introduce you to our new AI-powered learning app, designed to help you learn more efficiently and effectively.\n\nOur app uses cutting-edge technology to provide personalized learning experiences, tailored to your unique needs and learning style. With features like ad

# Scenario: AI-Powered Dev Workflow (Corporate Engineering Team)
🎯 Problem Statement
 In companies:
 - Developers manually write and test code.
 - Reviewers check for bugs and compliance.
 - Testing and validation are slow and error-prone.
 👉 Solution
 A multi-agent system powered by Groq API where:
 - 👨‍💻 Coder Agent writes Python code for requested features.
 - 🔍 Reviewer Agent validates logic and ensures best practices.
 - 🧪 Tester Agent generates unit tests and executes them automatically.
 - 🔁 The loop continues until the code passes all tests and meets requirements.
 🔄 Workflow
 Feature Request → Product Manager breaks into tasks → Coder writes code → Tester validates with unit tests → Reviewer checks compliance → Iteration loop → Final approved codebase
 🧹 Conflict Resolution
 - Resolve conflicting dependencies (e.g., package version mismatches).
 - Standardize coding style across agents (PEP8 compliance).
 - Ensure Groq API integration avoids latency or model mismatch issues.

In [16]:
# 🔹 Install (run once)
import sys
!{sys.executable} -m pip install pyautogen openai pytest

# 🔹 Imports
import autogen
from getpass import getpass

# 🔐 API Key
groq_api_key = getpass("Enter your GROQ API Key: ").strip()

# -----------------------------
# 🔧 LLM Config
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": groq_api_key,
            "base_url": "https://api.groq.com/openai/v1"
        }
    ],
    "temperature": 0
}

# -----------------------------
# 👨‍💼 Product Manager
# -----------------------------
pm = autogen.AssistantAgent(
    name="ProductManager",
    llm_config=llm_config,
    system_message="""
You are a Product Manager.

Break feature requests into clear development tasks.

Output:
- Step-by-step tasks
- Clear requirements
"""
)

# -----------------------------
# 👨‍💻 Coder
# -----------------------------
coder = autogen.AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message="""
You are a Python developer.

Write clean, modular, PEP8-compliant code.

Rules:
- Handle edge cases
- Add comments
- Ensure code is executable
- No syntax errors
"""
)

# -----------------------------
# 🧪 Tester
# -----------------------------
tester = autogen.AssistantAgent(
    name="Tester",
    llm_config=llm_config,
    system_message="""
You are a QA Engineer.

Tasks:
- Write pytest unit tests
- Cover edge cases
- Execute tests
- Report failures clearly
"""
)

# -----------------------------
# 🔍 Reviewer
# -----------------------------
reviewer = autogen.AssistantAgent(
    name="Reviewer",
    llm_config=llm_config,
    system_message="""
You are a Code Reviewer.

Check:
- Code correctness
- PEP8 compliance
- Performance issues
- Dependency conflicts

If everything is correct:
Return APPROVED

Else:
Suggest fixes
"""
)

# -----------------------------
# ⚙️ Executor
# -----------------------------
executor = autogen.UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config={
        "work_dir": "dev_project",
        "use_docker": False
    },
    is_termination_msg=lambda x: "APPROVED" in x.get("content", "")
)

# -----------------------------
# 👥 Group Chat
# -----------------------------
groupchat = autogen.GroupChat(
    agents=[executor, pm, coder, tester, reviewer],
    messages=[],
    max_round=12,
    speaker_selection_method="auto"
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

# -----------------------------
# 🚀 Feature Request
# -----------------------------
task = """
Build a Python module for a TODO application with:

Features:
- Add task
- Delete task
- Mark task as complete
- List all tasks

Requirements:
- Use clean class-based design
- Handle invalid inputs
- Write unit tests using pytest
- Follow PEP8

Run tests and iterate until APPROVED
"""

# -----------------------------
# ▶️ Run System
# -----------------------------
executor.initiate_chat(manager, message=task)

Enter your GROQ API Key: ··········
Executor (to chat_manager):


Build a Python module for a TODO application with:

Features:
- Add task
- Delete task
- Mark task as complete
- List all tasks

Requirements:
- Use clean class-based design
- Handle invalid inputs
- Write unit tests using pytest
- Follow PEP8

Run tests and iterate until APPROVED


--------------------------------------------------------------------------------
ProductManager (to chat_manager):

**TODO Application Module**

### Step 1: Design the Class Structure

*   Create a `Task` class to represent individual tasks
*   Create a `TodoList` class to manage the list of tasks

### Step 2: Implement the Task Class

*   Define the `Task` class with the following attributes:
    *   `id`: unique identifier for the task
    *   `description`: description of the task
    *   `completed`: boolean indicating whether the task is completed
*   Implement the `__init__` method to initialize the task attributes
*   Implement the `__

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kkdnrn2ff0d9ffbd668jtn39` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 96490, Requested 6498. Please try again in 43m1.632s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}